# 01 · Cost and budget — what a run spent, and the ceiling that stops it

Every other notebook in this repo wraps its paid calls in `nbio.cost_meter(...)`
and moves on. Nothing anywhere explains what that context manager is doing. This
notebook is that explanation: the rate table, the lookup, the accumulator, and —
the part worth the most — the two exceptions, fired for real rather than
described.

This is the **counting** level of observability: tokens, calls, dollars. It is the
cheapest of the three levels and the one that stops a mistake from becoming a
bill.

**No API key is needed, and none is used.** Every number below comes from
synthetic token counts fed straight into the meter. A budget check is arithmetic
over a usage block — it does not need a real call to be real.

## What this notebook demonstrates

| Name | What it does | One example |
|---|---|---|
| `nbio.PRICES` | USD per token, `(input, output)`, per model id | `PRICES["gpt-4o"] == (0.0000025, 0.00001)` |
| `nbio.price_for` | Rate for a model id, with a longest-key-contained fallback for wrapped ids | `price_for("vertex/gpt-4o-mini-2024") -> the gpt-4o-mini rate` |
| `nbio.Meter.record` | Adds one call's tokens, prices them, enforces the ceiling | `meter.record("gpt-4o-mini", 2000, 500)` |
| `nbio.Meter.record_cost` | Same ceiling when only a dollar figure is available, not tokens | `meter.record_cost("judge", 0.012)` |
| `nbio.BudgetExceeded` | Raised the moment accumulated spend reaches the ceiling | a 1000-iteration loop stops at call 84 |
| `nbio.UnpricedModel` | Raised when a budget is set but the model has no rate | `record("some-new-provider/frontier-v2", ...)` under a budget |
| `nbio.cost_meter` | The context manager that hands you a `Meter` with a ceiling | `with nbio.cost_meter(budget_usd=0.05) as meter:` |
| `Meter.report` | Per-model cost table, totals, and the ceiling, as text | printed at the end of most steps below |

## Step 1 — locate the repo root and import `nbio`

Jupyter starts a kernel with its working directory set to the notebook's own
folder, two levels below the repo root, so a bare `import nbio` fails. Walk up
until `nbio.py` is found, then import it — the same walk `nbio.bootstrap()` does
internally, which it cannot do for us because it has to be imported first.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — the rate table: what turns tokens into dollars

`nbio.PRICES` maps a model id to `(input_rate, output_rate)` in **USD per
token** — not per thousand tokens, which is how most provider pricing pages
quote it. Multiplying by the wrong factor of 1000 is the single easiest way to
set a ceiling that never fires, so the per-token unit is worth reading off the
table directly.

In [ ]:
# Per-token rates, shown at the per-million scale providers usually quote,
# so the two can be checked against each other.
rows = [
    (model, f"${rate_in * 1_000_000:,.3f}", f"${rate_out * 1_000_000:,.3f}")
    for model, (rate_in, rate_out) in list(nbio.PRICES.items())
]
nbio.table(rows, ("model id", "$ / 1M input", "$ / 1M output"))

print()
print(f"{len(nbio.PRICES)} models priced")
print("raw table entry for gpt-4o:", nbio.PRICES["gpt-4o"])

# The unit is dollars per token, not per thousand.
assert nbio.PRICES["gpt-4o"] == (0.00000250, 0.00001000)
assert nbio.PRICES["text-embedding-3-small"][1] == 0.0  # embeddings have no output side

## Step 3 — `price_for`: exact hit, wrapped id, and no hit at all

Providers wrap model names. The same model arrives as `gpt-4o-mini` from one
client and `vertex/gpt-4o-mini-2024-07-18` from another. `price_for` first tries
an exact key, then falls back to **the longest table key contained in the id** —
longest, not first, because `gpt-4o-mini-2024-07-18` contains both `gpt-4o` and
`gpt-4o-mini`, and the shorter match would price a cheap model at four times its
real rate.

In [ ]:
exact = nbio.price_for("gpt-4o-mini")
wrapped = nbio.price_for("vertex/gpt-4o-mini-2024-07-18")
unknown = nbio.price_for("some-new-provider/frontier-v2")
empty = nbio.price_for("")

print("exact    'gpt-4o-mini'                       ->", exact)
print("wrapped  'vertex/gpt-4o-mini-2024-07-18'     ->", wrapped)
print("unknown  'some-new-provider/frontier-v2'     ->", unknown)
print("empty    ''                                  ->", empty)

# Both substring keys are present in the wrapped id; the longer one has to win.
candidates = [k for k in nbio.PRICES if k in "vertex/gpt-4o-mini-2024-07-18"]
print()
print("keys contained in the wrapped id:", sorted(candidates))
print("shortest would have priced it as:", nbio.PRICES[min(candidates, key=len)])
print("longest (what price_for uses)   :", nbio.PRICES[max(candidates, key=len)])

assert wrapped == exact == nbio.PRICES["gpt-4o-mini"]
assert sorted(candidates) == ["gpt-4o", "gpt-4o-mini"]
assert nbio.PRICES[min(candidates, key=len)] != nbio.PRICES[max(candidates, key=len)]
assert unknown is None and empty is None

## Step 4 — what `Meter.record()` accumulates

A `Meter` holds five things: call count, prompt tokens, completion tokens,
total dollars, and dollars per model. `record(model_id, prompt_tokens,
completion_tokens)` adds one call to all of them, pricing it through
`price_for`. The token counts come from the provider's usage block on a real
call — here they are synthetic, which changes nothing about the arithmetic.

In [ ]:
meter = nbio.Meter()  # no ceiling yet: just counting

meter.record("openai/gpt-oss-120b", prompt_tokens=1200, completion_tokens=300)
meter.record("openai/gpt-oss-120b", prompt_tokens=900, completion_tokens=150)
meter.record("text-embedding-3-small", prompt_tokens=5000, completion_tokens=0)

# The same arithmetic, done by hand, to show there is nothing else in there.
gen_in, gen_out = nbio.PRICES["openai/gpt-oss-120b"]
emb_in, _ = nbio.PRICES["text-embedding-3-small"]
expected = (1200 + 900) * gen_in + (300 + 150) * gen_out + 5000 * emb_in

print(f"calls             : {meter.calls}")
print(f"prompt tokens     : {meter.prompt_tokens:,}")
print(f"completion tokens : {meter.completion_tokens:,}")
print(f"cost_usd          : ${meter.cost_usd:.6f}")
print(f"hand-computed     : ${expected:.6f}")
print()
print(meter.report())

assert meter.calls == 3
assert meter.prompt_tokens == 1200 + 900 + 5000
assert meter.completion_tokens == 300 + 150
assert abs(meter.cost_usd - expected) < 1e-12

## Step 5 — `BudgetExceeded`, fired for real on a runaway loop

This is the point of the whole file. The loop below asks for **1000** calls at
about six hundredths of a cent each, under a five-cent ceiling. It does not
finish. `record()` raises `BudgetExceeded` the moment cumulative spend reaches
the ceiling, and the loop dies there — caught, printed, and asserted below.

A ceiling that is only checked after the batch finishes is not a ceiling. This
one is checked on every call.

In [ ]:
PLANNED_CALLS = 1000
BUDGET = 0.05
completed = 0
caught = None

with nbio.cost_meter(budget_usd=BUDGET) as meter:
    try:
        for _ in range(PLANNED_CALLS):
            meter.record("gpt-4o-mini", prompt_tokens=2000, completion_tokens=500)
            completed += 1
    except nbio.BudgetExceeded as exc:
        caught = exc

print(f"asked for {PLANNED_CALLS} calls, got through {completed} before the stop")
print()
print(f"{type(caught).__name__}: {caught}")
print()
print(meter.report())

per_call = 2000 * nbio.PRICES["gpt-4o-mini"][0] + 500 * nbio.PRICES["gpt-4o-mini"][1]
print()
print(f"per call: ${per_call:.6f} — so the ceiling had to fall at call "
      f"{int(BUDGET / per_call) + 1}, and it did")

assert isinstance(caught, nbio.BudgetExceeded)
assert completed < PLANNED_CALLS
assert meter.calls == int(BUDGET / per_call) + 1
assert meter.cost_usd >= BUDGET
# What the loop would have cost had nothing stopped it:
assert PLANNED_CALLS * per_call > 10 * BUDGET

## Step 6 — the check runs *after* the call, not before it

The ceiling is enforced on the way out of `record()`, so a run stops **just
after** the call that crossed the line, never before it. Final spend is
therefore always a little over the budget, by at most one call's worth. If you
are metering against a hard external limit — a prepaid balance, a per-key cap —
set the ceiling under it by at least the price of your largest single call.

In [ ]:
overshoot = meter.cost_usd - BUDGET
print(f"ceiling      : ${BUDGET:.4f}")
print(f"final spend  : ${meter.cost_usd:.4f}")
print(f"overshoot    : ${overshoot:.6f}  (at most one call: ${per_call:.6f})")

assert meter.cost_usd >= BUDGET          # it did cross
assert 0 <= overshoot <= per_call        # by no more than the call that crossed it

## Step 7 — `record_cost`, for when you get dollars instead of tokens

Some callers never hand you a usage block. DeepEval's metrics, for example,
price their own judge call internally and expose `metric.evaluation_cost` after
`.measure()` — a float, no tokens. `record_cost` takes that float and puts it
under the same ceiling. Token counters stay at zero, because no tokens were ever
reported; the dollars are what is being enforced.

In [ ]:
judge_costs = [0.012, 0.012, 0.012]   # three judge calls, dollars only
caught_cost = None
scored = 0

with nbio.cost_meter(budget_usd=0.02) as judge_meter:
    try:
        for c in judge_costs:
            judge_meter.record_cost("deepeval-judge/gpt-4o-mini", c)
            scored += 1
    except nbio.BudgetExceeded as exc:
        caught_cost = exc

print(f"metrics scored before the stop: {scored} of {len(judge_costs)}")
print(f"(the second call is the one that crossed: 2 x $0.012 = $0.024 against a $0.02 ceiling)")
print(f"{type(caught_cost).__name__}: {caught_cost}")
print()
print(judge_meter.report())

assert isinstance(caught_cost, nbio.BudgetExceeded)
assert scored == 1                       # call 1 recorded; call 2 crossed and raised
assert judge_meter.calls == 2            # the crossing call is counted before the raise
assert judge_meter.prompt_tokens == 0    # no usage block was ever reported
assert judge_meter.cost_usd >= 0.02

## Step 8 — `UnpricedModel`, also fired for real

A model id with no entry in `PRICES` cannot be converted to dollars. Under a
budget, `record()` refuses: it raises `UnpricedModel` rather than charging
anything.

In [ ]:
caught_unpriced = None
with nbio.cost_meter(budget_usd=0.50) as strict:
    try:
        strict.record("some-new-provider/frontier-v2", prompt_tokens=50_000, completion_tokens=8_000)
    except nbio.UnpricedModel as exc:
        caught_unpriced = exc

print(f"{type(caught_unpriced).__name__}: {caught_unpriced}")
print()
print("meter state after the refusal:", strict.report())

assert isinstance(caught_unpriced, nbio.UnpricedModel)
assert strict.calls == 0        # the call was refused, not priced at zero
assert strict.cost_usd == 0.0

## Step 9 — why it is a hard stop and not a $0 fallback

The obvious alternative is to charge an unknown model $0 and carry on. Watch
what that costs you. With **no** budget set, `nbio`'s meter does exactly that —
it has no ceiling to protect, so an unpriced model is counted at zero and
flagged in the report. The same behaviour *under* a ceiling would be a disaster:
the one model whose cost you cannot see is the one that would burn through the
budget silently, and the ceiling would read `$0.0000` the whole way.

Put plainly: a silent $0 makes the ceiling unenforceable for exactly the model
that needs it most. An unknown rate under a budget is a hard stop, not a
fallback.

This is not a cookbook-only opinion. The private product's own meter,
`terrier_ta/services/llm_cost_meter.py`, fails closed the same way — its
`on_unpriced` defaults to `"error"` whenever a budget is set, and its docstring
records why: the project's old default model was deprecated and shut down by its
provider, a replacement was swapped in, and a meter that priced unknown models
at $0 would have reported a perfect $0.0000 run while spending real money.
`nbio`'s meter is the same rule with the options removed.

In [ ]:
def outcome(model_id, budget_usd):
    """What the meter does for one (model, budget) pair -- run, not described."""
    m = nbio.Meter(budget_usd=budget_usd)
    try:
        m.record(model_id, 50_000, 8_000)
        return f"recorded, ${m.cost_usd:.4f}"
    except nbio.UnpricedModel:
        return "refused: UnpricedModel"
    except nbio.BudgetExceeded:
        return f"stopped: BudgetExceeded at ${m.cost_usd:.4f}"


KNOWN, UNKNOWN = "gpt-4o-mini", "some-new-provider/frontier-v2"
grid = [
    ("priced model",   "no budget",   outcome(KNOWN, None)),
    ("priced model",   "$0.50",       outcome(KNOWN, 0.50)),
    ("priced model",   "$0.001",      outcome(KNOWN, 0.001)),
    ("unpriced model", "no budget",   outcome(UNKNOWN, None)),
    ("unpriced model", "$0.50",       outcome(UNKNOWN, 0.50)),
]
nbio.table(grid, ("model", "ceiling", "what the meter does"))

# The no-budget row is the $0 behaviour, visible in the report as "(unpriced)".
loose = nbio.Meter()
loose.record(UNKNOWN, 50_000, 8_000)
print()
print(loose.report())

assert loose.cost_usd == 0.0 and loose.calls == 1     # counted, priced at nothing
assert outcome(UNKNOWN, 0.50) == "refused: UnpricedModel"
assert outcome(KNOWN, 0.50).startswith("recorded")
assert outcome(KNOWN, 0.001).startswith("stopped")

## Step 10 — sizing a ceiling before you run anything

A budget picked out of the air either fires on a legitimate run or never fires
at all. The rate table lets you price the planned work first, from the shape of
the batch, and set the ceiling as a multiple of that estimate — high enough that
an honest run finishes, low enough that a loop bug or an off-by-1000 batch size
stops early.

Everything below is arithmetic over `PRICES`. No calls are made.

In [ ]:
def estimate_usd(model_id, n_calls, prompt_tokens, completion_tokens):
    """Priced from the table, before anything runs."""
    rate = nbio.price_for(model_id)
    if rate is None:
        raise nbio.UnpricedModel(f"{model_id!r} is unpriced -- cannot estimate")
    return n_calls * (prompt_tokens * rate[0] + completion_tokens * rate[1])


plan = [
    ("text-embedding-3-small", 40, 800, 0),      # embed 40 chunks
    ("openai/gpt-oss-120b", 20, 3_000, 400),     # score 20 candidates
    ("gpt-4o-mini", 1, 6_000, 900),              # write one answer
]
rows, total = [], 0.0
for model_id, n, pin, pout in plan:
    est = estimate_usd(model_id, n, pin, pout)
    total += est
    rows.append((model_id, n, f"{pin:,}/{pout:,}", f"${est:.5f}"))
nbio.table(rows, ("model", "calls", "tok in/out", "estimated"))

ceiling = round(total * 3, 2) or 0.01
print()
print(f"estimated run  : ${total:.5f}")
print(f"ceiling (3x)   : ${ceiling:.2f}  <- leaves room for a retry, stops a runaway")

# Run the plan against that ceiling: an honest run must finish under it.
with nbio.cost_meter(budget_usd=ceiling) as planned:
    for model_id, n, pin, pout in plan:
        for _ in range(n):
            planned.record(model_id, pin, pout)
print()
print(planned.report())

assert abs(planned.cost_usd - total) < 1e-9   # the estimate was the run
assert planned.cost_usd < ceiling              # nothing fired on the honest run
assert planned.calls == sum(n for _, n, _, _ in plan)

## Where this runs in the pipeline

Everywhere a paid call happens. `01-tools/03-embed`, `04-retrieve`, `05-gate`
and `06-bench` all open a `nbio.cost_meter(...)` block and call
`meter.record(model_id, prompt_tokens, completion_tokens)` with the usage block
the provider returned. This notebook is the one place that says what happens
next.

The counting level answers *what did it cost*. It cannot tell you what the run
produced (`02-run-artifacts.ipynb`) or why the run made the choices it did
(`03-tracing.ipynb`).

## What did not come across

- **Per-node itemization.** The product meter brackets each pipeline stage
  (`with meter.node("grade_attempt"):`) so spend can be read off stage by stage.
  `nbio`'s meter is deliberately flat — a notebook makes a handful of calls in
  one kernel, and per-node attribution would be machinery with nothing to
  attribute.
- **The per-call log.** The product keeps one `CallRecord` per call, which makes
  a saved run replayable call by call. `nbio` keeps only running totals per
  model. Step 10's plan table is the poor cousin of it.
- **Thread safety.** The product's meter holds a reentrant lock because grading
  nodes can run under `asyncio.to_thread`. `nbio`'s is single-threaded and
  unguarded; a notebook that fans calls out across threads needs the product
  version, not this one.
- **`on_unpriced="assume"`.** The product can charge a deliberately expensive
  fallback rate for an unknown model instead of refusing. `nbio` only has the
  refusal. Step 9's grid has no fourth row for it.
- **Cached input tokens.** Providers that serve part of a prompt from cache bill
  it at roughly half rate and report it separately. The product tracks this;
  `nbio` does not, so a cache-heavy run is over-estimated here.
- **Real usage blocks.** Everything above is synthetic token counts. On a real
  call the numbers arrive as `response.usage.prompt_tokens` /
  `.completion_tokens`, and providers occasionally return no usage block at all
  — which the product counts and flags, and `nbio` silently records as zero.